# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library and Croissant schema.

### Dataset Source
The dataset schema is provided as a Croissant JSON-LD file:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and contents using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
List the available record sets and their `@id` fields. Explore the fields for each record set using their respective IDs.

**Note:** The `mlcroissant` API allows introspection of `record_sets` and their structure. Each record set and field should be referenced by their `@id`.

In [ ]:
# List all available record sets with their @id and names using `dataset.info()`
record_sets_info = dataset.info()['record_sets']
print("Available record sets:")
for rs in record_sets_info:
    print(f"  - @id: {rs['@id']}, name: {rs.get('name', '[no name]')}")

# Save the list of record set @ids
record_set_ids = [rs['@id'] for rs in record_sets_info]

# For demonstration, print field @id and names for the first record set
if record_set_ids:
    example_rs_id = record_set_ids[0]
    print(f"\nFields for record set '@id': {example_rs_id}")
    for field in [f for f in record_sets_info if f['@id'] == example_rs_id][0]['fields']:
        print(f"  - @id: {field['@id']}, name: {field.get('name','[no name]')}")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for further analysis.
Here, each record set is referenced by its `@id`. All DataFrames are stored in a dictionary keyed by record set `@id`.

In [ ]:
# Read all record sets into dataframes
dfs = {}
for rs_id in record_set_ids:
    print(f"Extracting records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        dfs[rs_id] = pd.DataFrame(records)
        print(f"  Columns: {dfs[rs_id].columns.tolist()}")
        print(f"  Number of records: {len(dfs[rs_id])}")
    else:
        print("  No records found.")

# Print first rows of the first available dataframe
if dfs:
    first_rs_id = list(dfs.keys())[0]
    print(f"\nFirst few records in DataFrame for record set {first_rs_id}:")
    display(dfs[first_rs_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply typical EDA operations: select a numeric field (referenced by its `@id`), filter, normalize, and group as an example.

**Note:** Replace `<numeric_field_id>` and `<group_field_id>` with the exact field `@id`s as identified above for your concrete data.

In [ ]:
# Example: Pick the first DataFrame with data for EDA
if dfs:
    record_set_id = list(dfs.keys())[0]
    df = dfs[record_set_id]

    # Display available columns (field @id)
    print(f"Field @ids in record set {record_set_id}:")
    for col in df.columns:
        print(f"  - {col}")

    # Example selection: select first numeric-looking field
    import numpy as np
    numeric_field = None
    for col in df.columns:
        try:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
        except Exception:
            continue
    if numeric_field is None:
        # Try to coerce numerics
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_field = col
                    break
            except Exception:
                continue

    if numeric_field is not None:
        print(f"Selected numeric field @id: {numeric_field}")
        # Filter records where numeric_field > threshold (e.g. 10)
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field (z-score)
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized field {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by another field (pick first non-numeric column as group)
        group_field = None
        for col in df.columns:
            if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col]):
                group_field = col
                break

        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            grouped_df = grouped_df.rename(columns={numeric_field: f"mean_{numeric_field}"})
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found in this record set.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Generate a histogram of the normalized numeric field or a bar plot of means by group, if the previous analysis succeeded.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dfs and 'filtered_df' in locals() and not filtered_df.empty and numeric_field is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of normalized {numeric_field}")
    plt.xlabel(f"{numeric_field}_normalized")
    plt.ylabel("Count")
    plt.show()

    if 'grouped_df' in locals() and not grouped_df.empty:
        grouped_df = grouped_df.reset_index()
        plt.figure(figsize=(10, 5))
        sns.barplot(x=group_field, y=f"mean_{numeric_field}", data=grouped_df)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("No sufficient data for visualization.")

## 6. Conclusion
Using the `mlcroissant` library, we've loaded, examined, and analyzed the FAIR^2 dataset via its Croissant schema. We demonstrated how to reference all dataset elements by their `@id`, loaded available record sets, performed exploratory data analysis using numeric fields and groupings, and visualized distributions and group means for selected fields.

This workflow can be adapted to other Croissant-compliant datasets—simply substitute new schema URLs and use the record set and field `@id`s as identifiers.